# Data Cleaning

This notebook aligns all six raw datasets to a common weekly time index and assembles them into a single master dataframe (`master_weekly.csv`). Each source is resampled or forward-filled from its native frequency (daily, monthly, or annual) to weekly, then joined on the date index.

## Dependencies

Re-runs `data_pulling.ipynb` to bring all raw variables (`yf_data`, `cpi`, `unemployment`, `interest_rate`, `trends_df`, `fluview_df`, `wonder_df`, `cms_df`) into the current namespace.

In [1]:
%run data_pulling.ipynb

[*********************100%***********************]  4 of 4 completed


Price           Close                                         High             \
Ticker            KIE        PJP        XBI         XLV        KIE        PJP   
2010-01-04   8.816064  15.276187  17.483257   23.879538   8.830981  15.284270   
2010-01-05   8.922971  15.179203  17.550632   23.645338   8.940374  15.357021   
2010-01-06   8.945347  15.163037  17.784801   23.887091   8.960264  15.235781   
2010-01-07   9.082089  15.187282  17.784801   23.970179   9.094520  15.203446   
2010-01-08   9.057225  15.203443  17.865004   24.007961   9.087060  15.211525   
...               ...        ...        ...         ...        ...        ...   
2023-12-22  43.127811  73.477791  87.084122  129.506149  43.214259  73.545952   
2023-12-26  43.166229  73.867287  88.912842  129.793900  43.262284  73.974398   
2023-12-27  43.233467  74.149666  90.274475  130.340637  43.243074  74.285988   
2023-12-28  43.406357  74.334671  90.304291  130.676331  43.463990  74.558633   
2023-12-29  43.435177  74.10

## Resampling to Weekly Frequency

Each dataset arrives at a different native cadence. This section converts all of them to Sunday-ended ISO weeks (`W` offset) so they share a common index for joining.

### Yahoo Finance — Weekly Closing Prices and Volatility

Daily prices are resampled by taking the last closing price of each week. Weekly volatility is computed as the standard deviation of daily log-returns within each week.

In [2]:
import numpy as np
returns = np.log(yf_data['Close']).diff()
weekly_close = yf_data['Close'].resample('W').last()
weekly_vol = returns.resample('W').std()


weekly_vol.columns = [f"{col}_vol" for col in weekly_vol.columns]

print(weekly_close.shape)
print(weekly_close.head())
print(weekly_vol.head())

(730, 4)
Ticker           KIE        PJP        XBI        XLV
2010-01-10  9.057225  15.203443  17.865004  24.007961
2010-01-17  9.049768  15.413590  17.691778  24.340349
2010-01-24  8.721590  15.033717  17.666115  23.917307
2010-01-31  8.768827  15.049875  17.733477  23.630234
2010-02-07  8.656946  14.823567  17.617998  23.290281
             KIE_vol   PJP_vol   XBI_vol   XLV_vol
2010-01-10  0.008310  0.003637  0.005600  0.008326
2010-01-17  0.009688  0.010255  0.013182  0.009302
2010-01-24  0.018964  0.019509  0.016421  0.019749
2010-01-31  0.011464  0.007055  0.010720  0.005938
2010-02-07  0.022845  0.017174  0.027868  0.017661


The result is 730 weekly observations across 4 ETFs (2010-01-10 to 2023-12-31), each paired with a corresponding weekly volatility measure — the primary target variable for the analysis.

### FRED — Weekly Macroeconomic Indicators

FRED series are monthly so each week within a month carries the same value (forward-fill). Combining all three series into one dataframe simplifies the downstream join.

In [3]:
cpi_weekly = cpi.resample('W').ffill()
unemployment_weekly = unemployment.resample('W').ffill()
interest_rate_weekly = interest_rate.resample('W').ffill()

fred_weekly = pd.DataFrame({
    'cpi': cpi_weekly,
    'unemployment': unemployment_weekly,
    'interest_rate': interest_rate_weekly
})

print(fred_weekly.shape)
print(fred_weekly.head())

(858, 3)
                cpi  unemployment  interest_rate
2010-01-03  217.488           9.8           0.11
2010-01-10  217.488           9.8           0.11
2010-01-17  217.488           9.8           0.11
2010-01-24  217.488           9.8           0.11
2010-01-31  217.488           9.8           0.11


`fred_weekly` spans the full FRED history available for these three series, extending well past the 2010–2023 study window since FRED keeps publishing new monthly releases — the row count here grows on every re-run rather than staying fixed. The inner join with yfinance below trims this down to the shared 2010–2023 window. Forward-filling is appropriate here because macro indicators are published monthly and do not change between releases.

### Google Trends — Weekly Symptom Searches

Renames raw column headers to prefixed names to avoid collisions in the master dataframe and ensures the index is a proper DatetimeIndex.

In [4]:
trends_df.index = pd.to_datetime(trends_df.index)
trends_df.columns = ['trends_flu_symptoms', 'trends_fever', 'trends_shortness_of_breath']

print(trends_df.shape)
print(trends_df.head())

(732, 3)
            trends_flu_symptoms  trends_fever  trends_shortness_of_breath
date                                                                     
2009-12-27                   20            41                           3
2010-01-03                   16            45                           3
2010-01-10                   14            47                           2
2010-01-17                   13            45                           3
2010-01-24                   11            49                           3


### CDC FluView — Weekly ILI

FluView encodes dates as epidemiological week numbers (YYYYWW format). Converts epiweek integers to actual calendar dates using the epiweeks package's `.startdate()` method, which returns the ISO/CDC-convention Sunday start of each epiweek — avoiding the year-boundary bugs and Monday-misalignment that `%Y%W%w` string parsing produces. The result retains only the four columns needed for analysis.

In [5]:
from epiweeks import Week

fluview_df['date'] = fluview_df['epiweek'].astype(str).apply(
    lambda ew: Week(int(ew[:4]), int(ew[4:])).startdate()
)
fluview_df['date'] = pd.to_datetime(fluview_df['date'])
fluview_df = fluview_df.set_index('date')
fluview_weekly = fluview_df[['wili', 'ili', 'num_ili', 'num_patients']]
print(fluview_weekly.shape)
print(fluview_weekly.head())

(731, 4)
                wili       ili  num_ili  num_patients
date                                                 
2010-01-03  1.907118  1.982838    14299        721138
2010-01-10  1.867375  1.827486    14088        770895
2010-01-17  1.880723  1.926056    14757        766177
2010-01-24  1.969084  1.924947    15122        785580
2010-01-31  2.113868  2.088768    16037        767773


In [6]:
print(fluview_weekly.index[:5])
print(weekly_close.index[:5])

DatetimeIndex(['2010-01-03', '2010-01-10', '2010-01-17', '2010-01-24',
               '2010-01-31'],
              dtype='datetime64[ns]', name='date', freq=None)
DatetimeIndex(['2010-01-10', '2010-01-17', '2010-01-24', '2010-01-31',
               '2010-02-07'],
              dtype='datetime64[ns]', freq='W-SUN')


731 weekly ILI observations align almost exactly with the yfinance window, confirming no significant date gaps in the surveillance record.

### NCHS — Weekly Mortality (Forward-Filled from Annual)

NCHS publishes annual national death totals. Deaths are summed across all states and causes for each year, then the annual series is resampled to weekly and forward-filled so each week in a given year carries that year's total. This is an approximation — NCHS data will later be confirmed as a low-utility feature due to time-trend confounding.

In [7]:
# NCHS is annual data - convert year to datetime and ffill to weekly
# We'll aggregate deaths by year first (sum across all states and causes)
wonder_df['year'] = pd.to_datetime(wonder_df['year'], format='%Y')
wonder_df['deaths'] = pd.to_numeric(wonder_df['deaths'], errors='coerce')

# Group by year and sum total deaths
nchs_annual = wonder_df.groupby('year')['deaths'].sum()

# Resample to weekly and forward fill
nchs_weekly = nchs_annual.resample('W').ffill()

print(nchs_weekly.shape)
print(nchs_weekly.head())

(940,)
year
1999-01-03    8594450
1999-01-10    8594450
1999-01-17    8594450
1999-01-24    8594450
1999-01-31    8594450
Freq: W-SUN, Name: deaths, dtype: int64


940 weekly rows reflect the longer NCHS history (1999 onward); only 2010–2017 overlaps the study window because the CDC dataset used here cuts off at 2017.

## Data Inspection

Before merging, confirm the structure and usability of each dataset.

### CMS — Column Inventory

CMS inpatient data is cross-sectional (provider-level, not time-series), so it cannot be joined on a date index. Inspecting the columns confirms there is no year or date field to align it with the weekly master frame.

In [8]:
# CMS doesn't have a year column directly - check what we have
print(cms_df.columns.tolist())

['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']


CMS data contains provider identifiers, DRG codes, and payment averages but no temporal dimension suitable for a weekly join. It will be excluded from the master dataframe; its role in the project is descriptive rather than predictive.

## Aligning to a Common Weekly Anchor

In [9]:
weekly_close_r = weekly_close.copy()
weekly_vol_r = weekly_vol.copy()
fred_weekly_r = fred_weekly.copy()
trends_r = trends_df.copy()

fluview_r = fluview_weekly.copy()


nchs_r = nchs_weekly.copy()

Renames each cleaned series to its _r (reindexed) form for the master join. All six sources are already Sunday-anchored — five as a byproduct of `resample('W')`, which defaults to Sunday week-ends, and FluView natively via `epiweeks.startdate()` — so no shift is needed; each is copied through unchanged before the join.

## Building Master DataFrame

Joins all time-indexed datasets on their weekly date index. An inner join on the five time-series sources ensures every row has complete data for the primary variables. NCHS is left-joined because its history ends in 2017; missing post-2017 values are forward-filled from the last known annual total.

In [10]:
# Merge everything except NCHS first with inner join
master_df = weekly_close_r.copy()
master_df.index.name = 'date'

master_df = master_df.join(weekly_vol_r, how='inner')
master_df = master_df.join(fred_weekly_r, how='inner')
master_df = master_df.join(trends_r, how='inner')
master_df = master_df.join(fluview_r, how='inner')

# Join NCHS with outer then ffill so we don't lose post-2017 data
master_df = master_df.join(nchs_r.rename('nchs_deaths'), how='left')
master_df['nchs_deaths'] = master_df['nchs_deaths'].ffill()

print(master_df.shape)
print(master_df.index.min(), "to", master_df.index.max())
print(master_df.isnull().sum())

(730, 19)
2010-01-10 00:00:00 to 2023-12-31 00:00:00
KIE                           0
PJP                           0
XBI                           0
XLV                           0
KIE_vol                       0
PJP_vol                       0
XBI_vol                       0
XLV_vol                       0
cpi                           0
unemployment                  0
interest_rate                 0
trends_flu_symptoms           0
trends_fever                  0
trends_shortness_of_breath    0
wili                          0
ili                           0
num_ili                       0
num_patients                  0
nchs_deaths                   0
dtype: int64


The master dataframe has 730 rows and 19 columns, spanning 2010-01-10 to 2023-12-31. 

## Saving Processed Data

Writes the final master dataframe to `master_weekly.csv` in the working directory for use by the EDA and modeling notebooks.

In [11]:
master_df.to_csv('../data/processed/master_weekly.csv')
print("Saved.")

Saved.
